## SegmenAI - Customer Segmentation App

##### This is the notebook for a Customer Segmentation project

Load all necessary libraries

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

Load Customer Data

In [ ]:
df = pd.read_csv("customerdata.csv", parse_dates=['InvoiceDate'], encoding='ISO-8859-1')

Inspect Dataset

In [ ]:
df.head(5)

In [ ]:
df.info()

Create 'Total Amount' Column

In [ ]:
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']
df.head(5)

Save Latest Date in Dataset (For Calculating Recency Later)

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

Caculate & Create Columns for Recency, Frequency & Monetary Value

In [ ]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                   # Frequency
    'TotalAmount': 'sum'                                      # Monetary
}).reset_index()

Rename columns for Recency, Frequency & Monetary Value

In [ ]:
rfm.rename(columns={'InvoiceDate':'Recency','InvoiceNo':'Frequency','TotalAmount':'Monetary'}, inplace=True)
print(rfm.head())

Standardize Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

Perform Segmentation Using K-Means Clustering Algorithm

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Segment'] = kmeans.fit_predict(X_scaled)

Calculate Model's Sihouette Score

In [ ]:
score = silhouette_score(X_scaled, rfm['Segment'])
print(f"Silhouette Score: {score:.2f}")

Plot Segment Distribution

In [ ]:
ax = sns.countplot(x='Segment', data=rfm)

for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{int(height)}',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom')

ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.title("Customer Segments Distribution")
plt.xlabel("Segment")
plt.ylabel("Count")

plt.show()

Plot Monetary Contribution per Segment

In [ ]:
ax = rfm.groupby('Segment')['Monetary'].sum().plot(kind='bar')

for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{height:,.0f}',   
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom')
    
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.title("Total Monetary Value per Segment")
plt.xlabel("Segment")
plt.ylabel("Total Monetary")

plt.show()

Plot Average Recency per Segment

In [ ]:
ax = rfm.groupby('Segment')['Recency'].mean().plot(kind='bar')

for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{height:.1f}',   
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom')
    
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.title("Average Recency per Segment")
plt.xlabel("Segment")
plt.ylabel("Average Recency")

plt.show()

Plot Average Frequency per Segment

In [ ]:
ax = rfm.groupby('Segment')['Frequency'].mean().plot(kind='bar')

for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{height:.1f}',  
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom')
    
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.title("Average Frequency per Segment")
plt.xlabel("Segment")
plt.ylabel("Average Frequency")

plt.show()

Save Scaler & Model

In [ ]:
joblib.dump(kmeans, 'models/kmeans_rfm.pkl')
joblib.dump(scaler, 'models/scaler_rfm.pkl')

print("Model and scaler saved successfully!")